# 01 — Compute a Bounding Box

A **bounding box** (bbox) is the smallest axis-aligned rectangle that contains a set of coordinates.

It is one of the most useful spatial summaries you will encounter — not because it is precise, but because it is fast, simple, and good enough to do a lot of work before you need anything fancier.

This notebook shows you how to compute one from scratch.

## 1. What Is a Bounding Box?

A bounding box is the four outermost coordinate limits of a dataset:

```text
west  = minimum longitude  (leftmost)
south = minimum latitude   (bottommost)
east  = maximum longitude  (rightmost)
north = maximum latitude   (topmost)
```

Visually, it is the smallest rectangle (with sides parallel to the axes) that contains all the geometry.

```
    north ─────────────────────────
          |                       |
          |   your data here      |
          |                       |
    south ─────────────────────────
         west                   east
```

It is not magic. It is just four `min`/`max` calls over your coordinate values.

## 2. BBox Format Convention

Different tools represent bounding boxes differently. For this course we use the GeoJSON-aligned convention:

```python
[min_lon, min_lat, max_lon, max_lat]
#  west    south    east    north
```

This is also the format used by most spatial libraries (Shapely, Fiona, pyproj).

Some tools — especially mapping libraries — use `[min_lat, min_lon, max_lat, max_lon]` (lat first). You will need to swap when that comes up. For now, always default to `[min_lon, min_lat, max_lon, max_lat]` unless told otherwise.

## 3. Compute BBox from a List of Points

Start with the simplest case: a flat list of `[lon, lat]` coordinate pairs.

In [ ]:
# [longitude, latitude] — GeoJSON order
points = [
    [-98.5, 33.8],
    [-97.2, 34.1],
    [-99.0, 32.9],
    [-98.1, 33.4],
    [-97.8, 34.6],
]

# Separate the axes
lons = [p[0] for p in points]
lats = [p[1] for p in points]

# Compute the four limits
min_lon = min(lons)
min_lat = min(lats)
max_lon = max(lons)
max_lat = max(lats)

bbox = [min_lon, min_lat, max_lon, max_lat]
print(f"bbox: {bbox}")
print(f"  west  (min lon): {min_lon}")
print(f"  south (min lat): {min_lat}")
print(f"  east  (max lon): {max_lon}")
print(f"  north (max lat): {max_lat}")

## 4. Compute BBox from GeoJSON Point Features

Real data arrives as a `FeatureCollection`. The workflow is the same — you just need to extract coordinates from the nested structure first.

In [ ]:
fc_points = {
    "type": "FeatureCollection",
    "features": [
        {"type": "Feature", "geometry": {"type": "Point", "coordinates": [-98.5, 33.8]}, "properties": {"name": "A"}},
        {"type": "Feature", "geometry": {"type": "Point", "coordinates": [-97.2, 34.1]}, "properties": {"name": "B"}},
        {"type": "Feature", "geometry": {"type": "Point", "coordinates": [-99.0, 32.9]}, "properties": {"name": "C"}},
        {"type": "Feature", "geometry": {"type": "Point", "coordinates": [-98.1, 33.4]}, "properties": {"name": "D"}},
        {"type": "Feature", "geometry": {"type": "Point", "coordinates": [-97.8, 34.6]}, "properties": {"name": "E"}},
    ]
}

# Extract coordinate pairs from each Point feature
coords = [f["geometry"]["coordinates"] for f in fc_points["features"]]

lons = [c[0] for c in coords]
lats = [c[1] for c in coords]

bbox = [min(lons), min(lats), max(lons), max(lats)]
print(f"bbox: {bbox}")

## 5. Compute BBox from Polygon Coordinates

Polygons add one level of nesting. The `coordinates` field is a list of **rings** — the first is the exterior ring, any subsequent rings are holes. Each ring is a list of `[lon, lat]` pairs.

```python
polygon["coordinates"]           # list of rings
polygon["coordinates"][0]        # exterior ring
polygon["coordinates"][0][0]     # first vertex of exterior ring → [lon, lat]
```

To compute a bbox, flatten all the vertices from the exterior ring into a single list of coordinate pairs.

In [ ]:
polygon_feature = {
    "type": "Feature",
    "geometry": {
        "type": "Polygon",
        "coordinates": [[
            [-99.0, 33.5],
            [-97.2, 33.5],
            [-97.2, 34.8],
            [-99.0, 34.8],
            [-99.0, 33.5],   # closing vertex — same as the first
        ]]
    },
    "properties": {}
}

# Access the exterior ring (index 0)
exterior_ring = polygon_feature["geometry"]["coordinates"][0]

print(f"Exterior ring has {len(exterior_ring)} vertices (including closing vertex)")
for v in exterior_ring:
    print(f"  {v}")

lons = [v[0] for v in exterior_ring]
lats = [v[1] for v in exterior_ring]

bbox = [min(lons), min(lats), max(lons), max(lats)]
print(f"\nbbox: {bbox}")

## 6. Write a Reusable BBox Function

Rather than repeating the same four lines every time, wrap it in a function. The input is a flat list of `[lon, lat]` pairs. The output is a bbox list.

Then write a second function that handles the GeoJSON extraction — keeping the two concerns separate.

In [ ]:
def compute_bbox(coords):
    """
    Compute a bounding box from a list of [lon, lat] pairs.
    Returns [min_lon, min_lat, max_lon, max_lat].
    """
    lons = [c[0] for c in coords]
    lats = [c[1] for c in coords]
    return [min(lons), min(lats), max(lons), max(lats)]


def extract_point_coords(feature_collection):
    """
    Extract [lon, lat] coordinates from all Point features in a FeatureCollection.
    """
    return [
        f["geometry"]["coordinates"]
        for f in feature_collection["features"]
        if f["geometry"]["type"] == "Point"
    ]


def extract_polygon_coords(feature):
    """
    Extract [lon, lat] coordinates from a Polygon feature's exterior ring.
    """
    return feature["geometry"]["coordinates"][0]


# Test both paths
print("Point bbox:", compute_bbox(extract_point_coords(fc_points)))
print("Polygon bbox:", compute_bbox(extract_polygon_coords(polygon_feature)))

## 7. Interpreting the BBox

A bbox should be readable in plain English. After computing one, you should be able to describe it as a geographic region — not just a list of four numbers.

In [ ]:
def describe_bbox(bbox):
    """Print a bbox as labeled compass edges."""
    min_lon, min_lat, max_lon, max_lat = bbox
    print(f"  West  (min lon): {min_lon}")
    print(f"  South (min lat): {min_lat}")
    print(f"  East  (max lon): {max_lon}")
    print(f"  North (max lat): {max_lat}")
    print(f"  Width  (lon span): {max_lon - min_lon:.4f}°")
    print(f"  Height (lat span): {max_lat - min_lat:.4f}°")


bbox = compute_bbox(extract_point_coords(fc_points))
print(f"bbox: {bbox}")
describe_bbox(bbox)

## 8. Validate BBox Logic

A correctly computed bbox satisfies three conditions:

1. `min_lon <= max_lon` — the west edge is not east of the east edge
2. `min_lat <= max_lat` — the south edge is not north of the north edge
3. All four values fall within valid world ranges

If any of these fail, either the input data was bad or the computation was wrong. Check early.

In [ ]:
def validate_bbox(bbox):
    """
    Returns True if the bbox is internally consistent and within world bounds.
    Prints a message for each failure.
    """
    min_lon, min_lat, max_lon, max_lat = bbox
    valid = True

    if min_lon > max_lon:
        print(f"  ERROR: min_lon ({min_lon}) > max_lon ({max_lon})")
        valid = False
    if min_lat > max_lat:
        print(f"  ERROR: min_lat ({min_lat}) > max_lat ({max_lat})")
        valid = False
    if not (-180 <= min_lon <= 180 and -180 <= max_lon <= 180):
        print(f"  ERROR: longitude out of world range")
        valid = False
    if not (-90 <= min_lat <= 90 and -90 <= max_lat <= 90):
        print(f"  ERROR: latitude out of world range")
        valid = False

    if valid:
        print("  bbox is valid")
    return valid


# Good bbox
print("Test 1:")
validate_bbox([-99.0, 32.9, -97.2, 34.6])

# Flipped min/max — result of accidentally swapping lat/lon
print("\nTest 2 (swapped lat/lon input):")
validate_bbox([32.9, -99.0, 34.6, -97.2])

# Out-of-range
print("\nTest 3 (out of range):")
validate_bbox([-190.0, 32.9, -97.2, 34.6])

---

## Exercise A — Compute BBox Manually, Then Verify with Python

Given the coordinates below, first determine the bounding box by eye, then compute it with Python and confirm.

```python
coords = [
    [-104.9, 39.7],   # Denver
    [-118.2, 34.0],   # Los Angeles
    [-87.6,  41.8],   # Chicago
    [-73.9,  40.7],   # New York
    [-122.4, 37.7],   # San Francisco
]
```

What do you expect `min_lon`, `max_lon`, `min_lat`, `max_lat` to be before running the code?

In [ ]:
coords = [
    [-104.9, 39.7],
    [-118.2, 34.0],
    [-87.6,  41.8],
    [-73.9,  40.7],
    [-122.4, 37.7],
]

bbox = compute_bbox(coords)
print('Expected by eye:')
print('  min_lon = -122.4, max_lon = -73.9, min_lat = 34.0, max_lat = 41.8')
print('Computed bbox:', bbox)
validate_bbox(bbox)


## Exercise B — BBox from a GeoJSON FeatureCollection

Read the feature collection below and compute the bbox for **all Point features** using `compute_bbox` and `extract_point_coords`. Then validate it with `validate_bbox`.

In [ ]:
airfields = {
    'type': 'FeatureCollection',
    'features': [
        {'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [-97.37, 35.39]}, 'properties': {'name': 'Tinker AFB'}},
        {'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [-94.37, 35.34]}, 'properties': {'name': 'Fort Smith Regional'}},
        {'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [-101.70, 33.66]}, 'properties': {'name': 'Lubbock Preston Smith'}},
        {'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [-97.04, 32.85]}, 'properties': {'name': 'NAS Fort Worth JRB'}},
        {'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [-98.47, 29.53]}, 'properties': {'name': 'Kelly Field Annex'}},
    ]
}

coords = [f['geometry']['coordinates'] for f in airfields['features'] if f['geometry']['type'] == 'Point']
bbox = compute_bbox(coords)
print('BBox:', bbox)
validate_bbox(bbox)


## Exercise C — Per-Feature BBox Comparison

Compute a separate bbox for each feature in `airfields` and print each one. Which feature has the largest longitude span? (Hint: for Point features the bbox degenerates to a single point — that is expected.)

In [ ]:
largest_span = (-1, None)
for feature in airfields['features']:
    name = feature['properties']['name']
    bbox = compute_bbox([feature['geometry']['coordinates']])
    lon_span = bbox[2] - bbox[0]
    lat_span = bbox[3] - bbox[1]
    print(f"{name:<24} bbox={bbox} lon_span={lon_span:.2f} lat_span={lat_span:.2f}")
    if lon_span > largest_span[0]:
        largest_span = (lon_span, name)

print()
print('Largest longitude span:', largest_span[1], largest_span[0])
print('For Point features they all tie at 0.0 because each bbox collapses to a single coordinate.')


## Exercise D — Write `compute_bbox(points)` from Scratch

Without looking back at the earlier implementation, write `compute_bbox` again from scratch. Then test it on at least two different datasets — one point collection and one polygon's exterior ring.

In [ ]:
def compute_bbox(coords):
    lons = [p[0] for p in coords]
    lats = [p[1] for p in coords]
    return [min(lons), min(lats), max(lons), max(lats)]


# Test 1: point collection
test_points = [[-98.5, 33.8], [-97.2, 34.1], [-99.0, 32.9]]
print(compute_bbox(test_points))

# Test 2: polygon exterior ring
test_ring = [[-99.0, 33.5], [-97.2, 33.5], [-97.2, 34.8], [-99.0, 34.8], [-99.0, 33.5]]
print(compute_bbox(test_ring))


Your teammate is mixing up latitude and longitude fields inside the bbox. `29.53` is the south edge (`min_lat`), not the west edge. The actual west edge is `-97.04`, which is the minimum longitude in the bbox.


## Next

In [02 — Draw a Bounding Box](./02-Draw_BBox.ipynb), we convert the four numbers into a visible rectangle on an interactive map and use `fit_bounds` to zoom the view automatically.